# Spaceborne SAR Ship Wake Analysis — High-Speed GPU Pipeline
Trains ResNet-18, ResNet-34, and ResNet-18 3-Class models on GPU and packages all trained checkpoints and evaluation metrics for local deployment.

In [1]:
# 1. Clone repository and change directory
!git clone https://github.com/mohaimin2103001/thesis.git /kaggle/working/thesis
%cd /kaggle/working/thesis

Cloning into '/kaggle/working/thesis'...
remote: Enumerating objects: 73, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 73 (delta 5), reused 73 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (73/73), 40.85 MiB | 29.92 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/kaggle/working/thesis


In [2]:
# 2. Install dependencies
!pip install -q -r requirements.txt

In [3]:
# 3. Create target directory structure
!mkdir -p data/raw data/processed models outputs/figures outputs/metrics outputs/gradcam

In [4]:
# 4. Fast download of authentic 1.46 GB UEIKAP SAR dataset from Zenodo
!curl -L -C - "https://zenodo.org/records/18391027/files/UEIKAP_SAR.zip?download=1" -o data/raw/UEIKAP_SAR.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1498M  100 1498M    0     0  4108k      0  0:06:13  0:06:13 --:--:-- 3239k.9M    0     0  1751k      0  0:14:35  0:00:13  0:14:22 3097k0     0  4375k      0  0:05:50  0:00:42  0:05:08 5831k


In [5]:
# 5. Extract and normalize folder structure
!unzip -q data/raw/UEIKAP_SAR.zip -d data/raw/UEIKAP_SAR
!python -c "import os, shutil; raw='data/raw'; tgt=os.path.join(raw, 'UEIKAP_SAR'); [shutil.move(os.path.join(r, d), os.path.join(tgt, d)) for r, dirs, _ in os.walk(raw) if 'images' in dirs and r != tgt for d in dirs if not os.path.exists(os.path.join(tgt, d))]"

In [6]:
# 6. Generate AIS kinematic ground-truth labels
!python -m src.label_generator

Discovered 515 SAR image chips in /kaggle/working/thesis/data/raw/UEIKAP_SAR/images
Clean samples after kinematic filtering & buffer pruning: 464
Binary distribution: Straight=359, Turning=105
3-Class distribution: Straight=359, Left=44, Right=61

Partition breakdown:
  train: Total=277 | Straight=210 | Turning=67
  val  : Total= 93 | Straight= 70 | Turning=23
  test : Total= 94 | Straight= 79 | Turning=15

Saved master labeled dataset index to: /kaggle/working/thesis/data/processed/dataset_index.csv


In [7]:
# 7. Train Model 1: ResNet-18 Binary Classifier
!python -m src.train --arch resnet18 --mode binary --epochs 15 --batch_size 16

Initializing training for RESNET18 [BINARY] on device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 191MB/s]
Epoch [01/15] (10.00s) | Train Loss: 0.1435 Acc: 70.04% | Val Loss: 0.3488 Acc: 74.19% BalAcc: 49.29% F1: 0.4259
  --> Saved new best checkpoint with Val F1: 0.4259
Epoch [02/15] (8.23s) | Train Loss: 0.0955 Acc: 45.49% | Val Loss: 0.0874 Acc: 63.44% BalAcc: 47.98% F1: 0.4772
  --> Saved new best checkpoint with Val F1: 0.4772
Epoch [03/15] (8.14s) | Train Loss: 0.0812 Acc: 56.68% | Val Loss: 0.1290 Acc: 68.82% BalAcc: 47.17% F1: 0.4387
Epoch [04/15] (8.16s) | Train Loss: 0.0723 Acc: 61.73% | Val Loss: 0.1130 Acc: 72.04% BalAcc: 52.24% F1: 0.5093
  --> Saved new best checkpoint with Val F1: 0.5093
Epoch [05/15] (8.05s) | Train Loss: 0.0696 Acc: 60.29% | Val Loss: 0.0803 Acc: 56.99% BalAcc: 59.75% F1: 0.5419
  --> Sav

In [8]:
# 8. Train Model 2: ResNet-34 Polarimetric Model
!python -m src.train --arch resnet34 --mode binary --epochs 15 --batch_size 16

Initializing training for RESNET34 [BINARY] on device: cuda
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth
100%|███████████████████████████████████████| 83.3M/83.3M [00:00<00:00, 188MB/s]
Epoch [01/15] (8.79s) | Train Loss: 0.1105 Acc: 51.99% | Val Loss: 0.2264 Acc: 74.19% BalAcc: 49.29% F1: 0.4259
  --> Saved new best checkpoint with Val F1: 0.4259
Epoch [02/15] (8.28s) | Train Loss: 0.0838 Acc: 56.32% | Val Loss: 0.0752 Acc: 67.74% BalAcc: 62.52% F1: 0.6086
  --> Saved new best checkpoint with Val F1: 0.6086
Epoch [03/15] (8.38s) | Train Loss: 0.0783 Acc: 65.34% | Val Loss: 0.0826 Acc: 68.82% BalAcc: 54.47% F1: 0.5463
Epoch [04/15] (8.43s) | Train Loss: 0.0736 Acc: 60.65% | Val Loss: 0.1292 Acc: 72.04% BalAcc: 53.70% F1: 0.5321
Epoch [05/15] (8.36s) | Train Loss: 0.0798 Acc: 55.60% | Val Loss: 0.0785 Acc: 58.06% BalAcc: 61.93% F1: 0.5558
Epoch [06/15] (8.13s) | Train Loss: 0.0657 Acc: 58.48% | Val L

In [9]:
# 9. Train Model 3: ResNet-18 Directional 3-Class Model
!python -m src.train --arch resnet18 --mode 3class --epochs 15 --batch_size 16

Initializing training for RESNET18 [3CLASS] on device: cuda
Epoch [01/15] (8.31s) | Train Loss: 0.1390 Acc: 36.10% | Val Loss: 0.1098 Acc: 53.76% BalAcc: 39.19% F1: 0.3593
  --> Saved new best checkpoint with Val F1: 0.3593
Epoch [02/15] (8.01s) | Train Loss: 0.0985 Acc: 58.84% | Val Loss: 0.1135 Acc: 67.74% BalAcc: 44.62% F1: 0.4039
  --> Saved new best checkpoint with Val F1: 0.4039
Epoch [03/15] (7.84s) | Train Loss: 0.0967 Acc: 65.70% | Val Loss: 0.1200 Acc: 68.82% BalAcc: 39.60% F1: 0.4108
  --> Saved new best checkpoint with Val F1: 0.4108
Epoch [04/15] (8.09s) | Train Loss: 0.0908 Acc: 64.26% | Val Loss: 0.1243 Acc: 58.06% BalAcc: 43.19% F1: 0.4120
  --> Saved new best checkpoint with Val F1: 0.4120
Epoch [05/15] (8.16s) | Train Loss: 0.0831 Acc: 64.98% | Val Loss: 0.1321 Acc: 72.04% BalAcc: 42.34% F1: 0.4089
Epoch [06/15] (8.00s) | Train Loss: 0.0737 Acc: 68.95% | Val Loss: 0.1228 Acc: 69.89% BalAcc: 46.34% F1: 0.4547
  --> Saved new best checkpoint with Val F1: 0.4547
Epoch [0

In [10]:
# 10. Run Dual Ensemble with 4-fold Test-Time Augmentation (TTA)
!python -m src.ensemble_eval

Loading ResNet-18 and ResNet-34 checkpoints for Dual Ensemble...
Running Dual Ensemble with 4-fold TTA on 94 test samples...
DUAL-ARCHITECTURE ENSEMBLE (+ TTA) TEST PERFORMANCE:
  Overall Accuracy:      70.21%  (Baseline ResNet-18: 73.40%)
  Balanced Accuracy:     47.17%  (Baseline: 57.17%)
  Macro F1-Score:        0.4728  (Baseline: 0.5612)
  Specificity (TN Rate): 81.01%  (84.8% target)
  Sensitivity (TP Rate): 13.33%  (40.0% target)
  ROC-AUC:               0.4363
  Confusion Matrix:
[[64 15]
 [13  2]]
Ensemble confusion matrix saved to: /kaggle/working/thesis/outputs/figures/ensemble_confusion_matrix.png


In [11]:
# 11. Run Epistemic Uncertainty Quantification (MC-Dropout N=30)
!python -m src.uncertainty --samples 30

Loading checkpoint for Uncertainty Analysis: /kaggle/working/thesis/src/../models/best_resnet18_binary_model.pt
Executing 30 stochastic MC-Dropout forward passes across 94 test samples...
UNCERTAINTY QUANTIFICATION AUDIT:
  Overall Test Accuracy:                  60.64%
  Avg Predictive Entropy (Correct):       0.6330
  Avg Predictive Entropy (Incorrect):     0.6346
  Statistical Significance (Entropy Gap): +0.0016
Rejection curve saved to: /kaggle/working/thesis/outputs/figures/resnet18_uncertainty_rejection_curve.png


In [12]:
# 12. Generate Grad-CAM Hydrodynamic Heatmaps
!python -m src.explainability

Generating Grad-CAM hydrodynamic heatmaps for 10 test samples...
Grad-CAM visual attribution maps successfully saved to: /kaggle/working/thesis/outputs/gradcam


In [13]:
# 13. Evaluate Meteo-Marine Operational Domain Shift
!python -m src.domain_shift

Loading checkpoint for Domain Shift Stress-Testing: /kaggle/working/thesis/src/../models/best_resnet18_binary_model.pt
Running domain shift evaluations across severity levels...
Domain shift testing complete!
Domain shift curves saved to: /kaggle/working/thesis/outputs/figures/resnet18_domain_shift_curves.png


In [14]:
# 14. Qualitative Hydrodynamic Failure Analysis
!python -m src.failure_analysis

Loading checkpoint for Failure Analysis: /kaggle/working/thesis/src/../models/best_resnet18_binary_model.pt
Total test errors identified: 17 out of 94 samples.

Physical Failure Taxonomy Distribution:
  - Wake Dissipation / Ambiguous Temporal Latency: 15 samples (88.2%)
  - Pure Drift without Rudder Deflection (Lateral wind/current displacement): 2 samples (11.8%)
Failure analysis gallery saved to: /kaggle/working/thesis/outputs/figures/resnet18_failure_gallery.png


In [15]:
# 15. Package all results into a downloadable ZIP archive
!zip -r /kaggle/working/thesis_results.zip models/ outputs/ data/processed/
print('\nSUCCESS! Download thesis_results.zip from the Output tab on the right side of the screen.')

  adding: models/ (stored 0%)
  adding: models/best_resnet18_3class_model.pt (deflated 8%)
  adding: models/README.md (deflated 50%)
  adding: models/best_resnet18_binary_model.pt (deflated 8%)
  adding: models/best_resnet34_binary_model.pt (deflated 8%)
  adding: outputs/ (stored 0%)
  adding: outputs/figures/ (stored 0%)
  adding: outputs/figures/resnet34_binary_training_curves.png (deflated 12%)
  adding: outputs/figures/ensemble_confusion_matrix.png (deflated 25%)
  adding: outputs/figures/resnet18_3class_training_curves.png (deflated 13%)
  adding: outputs/figures/resnet18_uncertainty_rejection_curve.png (deflated 14%)
  adding: outputs/figures/resnet18_binary_training_curves.png (deflated 13%)
  adding: outputs/figures/preprocessing_comparison.png (deflated 10%)
  adding: outputs/figures/resnet18_failure_gallery.png (deflated 9%)
  adding: outputs/figures/resnet18_domain_shift_curves.png (deflated 13%)
  adding: outputs/gradcam/ (stored 0%)
  adding: outputs/gradcam/gradcam_00_St